# 🧠 DeepFake Detector - Entrenamiento con GPU

Este notebook entrena el modelo **DenseNet-121** para detección de deepfakes usando **GPU gratuita** de Google Colab.

### 📋 Instrucciones:
1. **Sube tu proyecto `detectorIA`** a Google Drive (ver paso 1)
2. **Ejecuta las celdas en orden** (da clic en ▶ y presiona Shift+Enter)
3. **Descarga el modelo entrenado** al finalizar (ver paso final)

### ⚠️ IMPORTANTE:
- Tu código local **NO se borra ni se modifica**
- Solo usaremos Colab para entrenar más rápido con GPU
- El modelo entrenado se guardará en tu Google Drive

---

## 📁 Paso 1: Subir tu proyecto a Google Drive

**Antes de ejecutar este notebook**, necesitas:

1. Ir a [Google Drive](https://drive.google.com)
2. Crear una carpeta llamada `detectorIA` en la raíz de tu Drive
3. **Subir todos los archivos** de tu proyecto a esa carpeta
   - Subcarpetas: `src/`, `app/`
   - Archivos: `run_pipeline.py`, `requirements.txt`, `train_quick.py`, etc.

> 💡 Puedes arrastrar y soltar los archivos directamente en Google Drive

**Estructura esperada en Google Drive:**
```
Mi unidad/
└── detectorIA/
    ├── src/
    │   ├── config.py
    │   ├── model.py
    │   ├── data_pipeline.py
    │   ├── train.py
    │   ├── evaluate.py
    │   ├── utils.py
    │   └── __init__.py
    ├── app/
    ├── run_pipeline.py
    ├── train_quick.py
    └── requirements.txt
```

## 🔧 Paso 2: Verificar GPU y conectar Drive

In [ ]:
# Verificar que hay GPU disponible
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"GPU disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memoria GPU: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
else:
    print("\n⚠️ NO hay GPU disponible. Ve a Runtime > Change runtime type y selecciona GPU.")
    print("  (En el menú: Runtime > Cambiar tipo de entorno de ejecución > GPU)")

In [ ]:
# Conectar Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("\n✅ Google Drive conectado!")

## 📦 Paso 3: Instalar dependencias

In [ ]:
# Instalar dependencias necesarias
!pip install -q torch torchvision pillow opencv-python-headless numpy matplotlib seaborn scikit-learn tqdm pandas kagglehub

print("✅ Dependencias instaladas!")

## 🔑 Paso 4: Configurar API de Kaggle (para descargar el dataset)

Necesitas una cuenta de Kaggle y tu API key:

1. Ve a [kaggle.com](https://www.kaggle.com)
2. Inicia sesión (o crea cuenta gratis)
3. Ve a tu perfil → Account → Create New API Token
4. Se descargará un archivo `kaggle.json`
5. **Copia tu `username` y `key`** de ese archivo

> 📄 El archivo `kaggle.json` se ve así:
> ```json
> {"username": "tu_usuario", "key": "tu_api_key_aqui"}
> ```

In [ ]:
# Configurar Kaggle API
# Reemplaza los valores entre comillas con tus datos reales

import os

# ⬇️ PEGA TU USERNAME Y KEY AQUÍ (entre las comillas)
KAGGLE_USERNAME = "TU_USERNAME_AQUI"  # <-- cambia esto
KAGGLE_KEY = "TU_KEY_AQUI"           # <-- cambia esto

# Configurar variables de entorno
os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
os.environ['KAGGLE_KEY'] = KAGGLE_KEY

# Crear archivo kaggle.json para kagglehub
import json
kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
with open(os.path.join(kaggle_dir, 'kaggle.json'), 'w') as f:
    json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
os.chmod(os.path.join(kaggle_dir, 'kaggle.json'), 0o600)

print("✅ API de Kaggle configurada!")

## 📂 Paso 5: Configurar el proyecto

In [ ]:
# Configurar ruta del proyecto
PROJECT_DIR = '/content/drive/MyDrive/detectorIA'  # <-- Verifica que esta ruta sea correcta

import sys
sys.path.insert(0, PROJECT_DIR)

# Verificar que el proyecto existe
import os
if os.path.exists(PROJECT_DIR):
    print(f"✅ Proyecto encontrado en: {PROJECT_DIR}")
    print(f"   Contenido: {os.listdir(PROJECT_DIR)}")
    if os.path.exists(os.path.join(PROJECT_DIR, 'src')):
        print(f"   src/: {os.listdir(os.path.join(PROJECT_DIR, 'src'))}")
else:
    print(f"❌ No se encontró el proyecto en: {PROJECT_DIR}")
    print("   Verifica que subiste la carpeta detectorIA a Google Drive")
    print("   La estructura debe ser: Mi unidad/detectorIA/src/...")

In [ ]:
# Cambiar al directorio del proyecto
os.chdir(PROJECT_DIR)
print(f"Directorio actual: {os.getcwd()}")

## 📥 Paso 6: Descargar y preparar el dataset

In [ ]:
# Descargar dataset 140K Real and Fake Faces
from src.data_pipeline import run_pipeline

print("📥 Descargando dataset desde Kaggle...")
print("   Esto puede tomar 5-15 minutos dependiendo de tu conexión.")
print("\n")

run_pipeline(download=True)

print("\n✅ Dataset listo!")

## 🚀 Paso 7: Entrenar el modelo con GPU

Con GPU, el entrenamiento completo (~140K imágenes, 30 épocas) tardará **~20-40 minutos** en lugar de horas.

In [ ]:
# Entrenar el modelo completo con GPU
import torch
print(f"🎮 Usando dispositivo: {'GPU (' + torch.cuda.get_device_name(0) + ')' if torch.cuda.is_available() else 'CPU'}")
print("\n")

from src.train import train

# Entrenar con configuración completa
history, test_metrics = train(
    epochs=30,           # Épocas completas
    batch_size=32,       # Batch size óptimo para GPU T4
    num_workers=2,       # Workers para Colab
    freeze_until_block=3 # Descongelar bloque 4 + clasificador
)

print("\n" + "=" * 60)
print("  ✅ ENTRENAMIENTO COMPLETADO!")
print("=" * 60)
print(f"  Test Accuracy: {test_metrics['metrics']['accuracy']:.4f}")
print(f"  Test AUC: {test_metrics['metrics']['auc']:.4f}")

## 📊 Paso 8: Ver resultados

In [ ]:
# Ver gráficos de entrenamiento
from IPython.display import Image, display

plots_dir = os.path.join(PROJECT_DIR, 'outputs', 'plots')

print("📊 Gráficos de entrenamiento:")
for img_name in ['training_history.png', 'confusion_matrix.png', 'roc_curve.png']:
    img_path = os.path.join(plots_dir, img_name)
    if os.path.exists(img_path):
        print(f"\n--- {img_name} ---")
        display(Image(filename=img_path, width=600))
    else:
        print(f"⚠️ {img_name} no encontrado")

In [ ]:
# Ver métricas finales
import json

metrics_path = os.path.join(plots_dir, 'final_metrics.json')
if os.path.exists(metrics_path):
    with open(metrics_path, 'r') as f:
        metrics = json.load(f)
    
    print("\n📋 Métricas finales en conjunto de prueba:")
    print("=" * 45)
    for key, value in metrics.items():
        if isinstance(value, float):
            print(f"  {key:<20} {value:.4f}")
        else:
            print(f"  {key:<20} {value}")
    print("=" * 45)
else:
    print("⚠️ Métricas no encontradas")

## 💾 Paso 9: El modelo se guardó automáticamente

Los checkpoints del modelo se guardaron en Google Drive en:
```
Mi unidad/detectorIA/outputs/checkpoints/
├── best_model.pth    (mejor modelo)
└── last_model.pth    (última época)
```

### 📥 Para descargar el modelo a tu computadora:

1. Ve a [Google Drive](https://drive.google.com)
2. Navega a `Mi unidad/detectorIA/outputs/checkpoints/`
3. Descarga `best_model.pth`
4. Copia el archivo en tu carpeta local: `detectorIA/outputs/checkpoints/`

> 💡 **Tu código local queda intacto.** Solo estás agregando el modelo entrenado.

In [ ]:
# Verificar que los checkpoints se guardaron
checkpoints_dir = os.path.join(PROJECT_DIR, 'outputs', 'checkpoints')

print("💾 Checkpoints guardados en Google Drive:")
print(f"   Ruta: {checkpoints_dir}")
print()

if os.path.exists(checkpoints_dir):
    for f in os.listdir(checkpoints_dir):
        size_mb = os.path.getsize(os.path.join(checkpoints_dir, f)) / (1024 * 1024)
        print(f"   📦 {f} ({size_mb:.1f} MB)")
else:
    print("   ⚠️ La carpeta de checkpoints no se creó")

print("\n")
print("=" * 60)
print("  📋 RESUMEN")
print("=" * 60)
print("  1. El modelo está en tu Google Drive")
print("  2. Descarga best_model.pth a tu computadora")
print("  3. Pégalo en: detectorIA/outputs/checkpoints/")
print("  4. Tu código local NO se modificó")
print("  5. Puedes ejecutar la app con: python run_pipeline.py --mode app")
print("=" * 60)

---

## 🎯 ¿Qué sigue después?

Una vez que descargues `best_model.pth` a tu computadora:

1. **Probar la app Streamlit:**
   ```bash
   python run_pipeline.py --mode app
   ```

2. **Evaluar el modelo:**
   ```bash
   python run_pipeline.py --mode evaluate
   ```

3. **Pruebas de robustez:**
   ```bash
   python run_pipeline.py --mode robustness
   ```

---

## ❓ Preguntas frecuentes

**¿Borré algo de mi computadora?**
→ NO. Tu proyecto local está intacto. Solo subiste una copia a Colab.

**¿Puedo volver a entrenar?**
→ Sí, simplemente ejecuta las celdas de nuevo en Colab.

**¿Cuánto cuesta?**
→ Nada. La GPU T4 de Colab es gratuita.

**¿Mis datos quedan en Colab?**
→ Solo en tu Google Drive. Colab no almacena datos permanentemente.